# Data Preprocessing & Exploratory Analysis

## Comparative Sentiment Analysis of Biden vs. Trump Tweets

This notebook consolidates data preprocessing and exploratory data analysis (EDA) for a comparative sentiment analysis project. We process raw tweet datasets for both Biden and Trump, clean the text, and perform initial exploratory analysis including word frequency, n-grams, topic modeling, and sentiment scoring with TextBlob and VADER.

**Project Overview:**
- **02_roberta_sentiment_analysis.ipynb**: Advanced transformer-based sentiment analysis using RoBERTa
- **03_comparative_analysis.ipynb**: Comparative analysis and visualizations across both datasets

In [ ]:
!pip install -q nrclex pyLDAvis gensim

import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from textblob import TextBlob

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('vader_lexicon', quiet=True)

print("All dependencies installed and NLTK data downloaded.")

In [ ]:
# -- Configuration --
TRUMP_RAW_PATH = "/content/drive/MyDrive/hashtag_donaldtrump.csv"
BIDEN_RAW_PATH = "/content/drive/MyDrive/hashtag_joebiden.csv"

from google.colab import drive
drive.mount("/content/drive")

data_trump = pd.read_csv(TRUMP_RAW_PATH, lineterminator="\n")
data_biden = pd.read_csv(BIDEN_RAW_PATH, lineterminator="\n")

print(f"Trump dataset shape: {data_trump.shape}")
print(f"Biden dataset shape: {data_biden.shape}")
print(f"\nTrump columns: {list(data_trump.columns)}")
print(f"Biden columns: {list(data_biden.columns)}")

## Text Preprocessing

We apply a 4-step preprocessing pipeline to clean tweet text:

1. **Lowercase Conversion**: Convert all text to lowercase for consistency
2. **URL Removal**: Remove URLs (http/https links)
3. **Special Character Removal**: Remove punctuation, symbols, and non-word characters
4. **Digit Removal**: Remove numeric digits to focus on textual content

This preprocessing standardizes the data for downstream analysis and modeling.

In [ ]:
def preprocess_tweets(df):
    """
    Clean tweet text through a 4-step pipeline:
    1. Lowercase
    2. Remove URLs
    3. Remove special characters
    4. Remove digits
    """
    df = df.copy()
    
    # 1. Lowercase
    df['tweet'] = df['tweet'].apply(lambda x: x.lower() if isinstance(x, str) else x)
    
    # 2. Remove URLs
    url_pattern = re.compile(r'https?://\S+')
    df['tweet'] = df['tweet'].apply(lambda x: url_pattern.sub('', x) if isinstance(x, str) else x)
    
    # 3. Remove non-word/non-whitespace characters (punctuation, symbols)
    df['tweet'] = df['tweet'].replace(to_replace=r'[^\w\s]', value='', regex=True)
    
    # 4. Remove digits
    df['tweet'] = df['tweet'].replace(to_replace=r'\d', value='', regex=True)
    
    return df

# Apply preprocessing to both datasets
data_trump = preprocess_tweets(data_trump)
data_biden = preprocess_tweets(data_biden)

print("Preprocessing complete.")
print(f"\nTrump sample tweets (first 3):")
for i, tweet in enumerate(data_trump['tweet'].head(3).tolist(), 1):
    print(f"  {i}. {tweet[:80]}...")

print(f"\nBiden sample tweets (first 3):")
for i, tweet in enumerate(data_biden['tweet'].head(3).tolist(), 1):
    print(f"  {i}. {tweet[:80]}...")

In [ ]:
# Save preprocessed datasets for reuse
data_trump.to_csv('preprocessed_trump.csv', index=False)
data_biden.to_csv('preprocessed_biden.csv', index=False)
print("Preprocessed datasets saved:")
print("  - preprocessed_trump.csv")
print("  - preprocessed_biden.csv")

## Exploratory Data Analysis

We perform comprehensive EDA to understand the structure and characteristics of both datasets, including text length distributions, word statistics, and frequency analysis.

In [ ]:
# Text Length Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data_trump['tweet'].str.len().hist(ax=axes[0], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0].set_title("Tweet Length Distribution -- Trump", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Character Count")
axes[0].set_ylabel("Frequency")
axes[0].grid(axis='y', alpha=0.3)

data_biden['tweet'].str.len().hist(ax=axes[1], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[1].set_title("Tweet Length Distribution -- Biden", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Character Count")
axes[1].set_ylabel("Frequency")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTrump tweets - Length stats:")
print(f"  Mean: {data_trump['tweet'].str.len().mean():.1f}, Median: {data_trump['tweet'].str.len().median():.1f}")
print(f"\nBiden tweets - Length stats:")
print(f"  Mean: {data_biden['tweet'].str.len().mean():.1f}, Median: {data_biden['tweet'].str.len().median():.1f}")

In [ ]:
# Average Word Length Distribution
def calc_avg_word_length(text):
    """Calculate average word length for a text."""
    words = str(text).split()
    if len(words) == 0:
        return 0
    return np.mean([len(w) for w in words])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trump_word_lengths = data_trump['tweet'].apply(calc_avg_word_length)
trump_word_lengths.hist(ax=axes[0], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0].set_title("Avg Word Length Distribution -- Trump", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Characters per Word")
axes[0].set_ylabel("Frequency")
axes[0].grid(axis='y', alpha=0.3)

biden_word_lengths = data_biden['tweet'].apply(calc_avg_word_length)
biden_word_lengths.hist(ax=axes[1], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[1].set_title("Avg Word Length Distribution -- Biden", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Characters per Word")
axes[1].set_ylabel("Frequency")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTrump word length - Mean: {trump_word_lengths.mean():.2f}, Median: {trump_word_lengths.median():.2f}")
print(f"Biden word length - Mean: {biden_word_lengths.mean():.2f}, Median: {biden_word_lengths.median():.2f}")

## Word Frequency Analysis

We extract the most common words from each dataset, excluding English stopwords and short words (length less than or equal to 2 characters).

In [ ]:
# Initialize stopwords
stop = set(stopwords.words('english'))

def get_top_words(df, col='tweet', n=30):
    """
    Extract top n non-stopword tokens from a DataFrame column.
    Filters out stopwords and words with length <= 2.
    """
    corpus = []
    for tokens in df[col].str.split().dropna():
        corpus.extend([w for w in tokens if w not in stop and len(w) > 2])
    return Counter(corpus).most_common(n)

# Generate and plot top words
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

datasets = [
    (data_trump, "Trump", '#e74c3c'),
    (data_biden, "Biden", '#3498db')
]

for ax, (data, label, color) in zip(axes, datasets):
    top = get_top_words(data, n=30)
    words, counts = zip(*top)
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.7, edgecolor='black')
    ax.set_title(f"Top 30 Words -- {label}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Frequency")
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## N-gram Exploration

We extract bigrams (2-grams) and trigrams (3-grams) to identify common word pairs and phrases in each dataset.

In [ ]:
def get_top_ngrams(corpus, n=2, top_k=15):
    """
    Extract top-k n-grams from a list of documents.
    Uses sklearn's CountVectorizer for efficient n-gram extraction.
    """
    vec = CountVectorizer(ngram_range=(n, n), max_features=top_k).fit(corpus)
    bow = vec.transform(corpus)
    sum_words = bow.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    return sorted(words_freq, key=lambda x: x[1], reverse=True)

# Generate and plot n-grams
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

datasets = [
    (data_trump, "Trump"),
    (data_biden, "Biden")
]

for col, (data, label) in enumerate(datasets):
    tweets = data['tweet'].dropna().tolist()
    for row, n in enumerate([2, 3]):
        ngrams = get_top_ngrams(tweets, n=n, top_k=15)
        words, counts = zip(*ngrams)
        axes[row][col].barh(words[::-1], counts[::-1], alpha=0.7, edgecolor='black')
        axes[row][col].set_title(f"Top {n}-grams -- {label}", fontsize=11, fontweight='bold')
        axes[row][col].set_xlabel("Frequency")
        axes[row][col].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## LDA Topic Modeling

We apply Latent Dirichlet Allocation (LDA) to identify latent topics in each dataset. LDA reveals thematic structure by discovering word clusters that frequently co-occur.

In [ ]:
import gensim.corpora
import gensim.models
import pyLDAvis
import pyLDAvis.gensim_models

def run_lda(df, col='tweet', num_topics=4, passes=10):
    """
    Run LDA topic modeling on a DataFrame's text column.
    
    Parameters:
    - df: DataFrame
    - col: Column name containing text
    - num_topics: Number of topics to extract
    - passes: Number of passes through the corpus
    
    Returns:
    - lda_model: Trained LDA model
    - bow_corpus: Bag-of-words representation
    - dictionary: Gensim Dictionary object
    """
    lem = WordNetLemmatizer()
    
    # Tokenize and lemmatize
    corpus_tokens = []
    for text in df[col].dropna():
        words = [lem.lemmatize(w) for w in word_tokenize(str(text)) 
                 if w not in stop and len(w) > 2]
        corpus_tokens.append(words)
    
    # Create dictionary and bag-of-words corpus
    dictionary = gensim.corpora.Dictionary(corpus_tokens)
    bow_corpus = [dictionary.doc2bow(doc) for doc in corpus_tokens]
    
    # Train LDA model
    lda_model = gensim.models.LdaMulticore(
        bow_corpus,
        num_topics=num_topics,
        id2word=dictionary,
        passes=passes,
        workers=2
    )
    
    return lda_model, bow_corpus, dictionary

# Run LDA on Trump tweets
print("Running LDA on Trump tweets (4 topics, 10 passes)...")
lda_trump, bow_trump, dict_trump = run_lda(data_trump, num_topics=4, passes=10)
print("\nTrump Topics:")
for idx, topic in lda_trump.print_topics(-1, num_words=8):
    print(f"  Topic {idx}: {topic}")

# Run LDA on Biden tweets
print("\n" + "="*60)
print("Running LDA on Biden tweets (4 topics, 10 passes)...")
lda_biden, bow_biden, dict_biden = run_lda(data_biden, num_topics=4, passes=10)
print("\nBiden Topics:")
for idx, topic in lda_biden.print_topics(-1, num_words=8):
    print(f"  Topic {idx}: {topic}")

### Interactive Topic Cluster Visualization

We use pyLDAvis to create interactive visualizations of the LDA topic models. Each topic is represented as a circle in 2D space (via PCA), where:
- **Circle size** reflects the topic's prevalence in the corpus
- **Distance between circles** indicates how different topics are from one another
- **Right panel** shows the most relevant terms for the selected topic

In [ ]:
# Interactive topic cluster visualization — Trump
pyLDAvis.enable_notebook()
trump_vis = pyLDAvis.gensim_models.prepare(lda_trump, bow_trump, dict_trump)
print("Trump Topic Clusters:")
trump_vis

In [ ]:
# Interactive topic cluster visualization — Biden
biden_vis = pyLDAvis.gensim_models.prepare(lda_biden, bow_biden, dict_biden)
print("Biden Topic Clusters:")
biden_vis

## TextBlob Sentiment Analysis

We apply TextBlob's polarity-based sentiment analysis to obtain a continuous sentiment score (-1 to +1) for each tweet, then classify as Positive, Negative, or Neutral.

In [ ]:
def get_polarity(text):
    """Calculate TextBlob polarity score."""
    try:
        return TextBlob(str(text)).sentiment.polarity
    except:
        return 0

def polarity_label(score):
    """Convert polarity score to label."""
    if score > 0:
        return 'Positive'
    elif score < 0:
        return 'Negative'
    return 'Neutral'

# Calculate polarity scores
for data, label in [(data_trump, "Trump"), (data_biden, "Biden")]:
    data['polarity_score'] = data['tweet'].apply(get_polarity)
    data['polarity'] = data['polarity_score'].apply(polarity_label)

# Plot sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets = [
    (data_trump, "Trump", '#e74c3c'),
    (data_biden, "Biden", '#3498db')
]

for ax, (data, label, color) in zip(axes, datasets):
    sentiment_counts = data['polarity'].value_counts()
    sentiment_counts = sentiment_counts.reindex(['Positive', 'Negative', 'Neutral'], fill_value=0)
    sentiment_counts.plot(kind='bar', ax=ax, color=color, alpha=0.7, edgecolor='black')
    ax.set_title(f"TextBlob Sentiment Distribution -- {label}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Sentiment")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTrump - TextBlob Sentiment:")
print(data_trump['polarity'].value_counts())
print(f"\nBiden - TextBlob Sentiment:")
print(data_biden['polarity'].value_counts())

## VADER Sentiment Analysis

We apply VADER (Valence Aware Dictionary and sEntiment Reasoner), a lexicon-based sentiment tool optimized for social media text. VADER returns a compound score (-1 to +1) that combines positive, negative, and neutral sentiment intensities.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

def vader_compound(text):
    """Extract VADER compound sentiment score."""
    try:
        return sid.polarity_scores(str(text))['compound']
    except:
        return 0

# Calculate VADER scores
for data, label in [(data_trump, "Trump"), (data_biden, "Biden")]:
    data['vader_score'] = data['tweet'].apply(vader_compound)

# Plot VADER compound score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets = [
    (data_trump, "Trump", '#e74c3c'),
    (data_biden, "Biden", '#3498db')
]

for ax, (data, label, color) in zip(axes, datasets):
    data['vader_score'].hist(ax=ax, bins=50, color=color, alpha=0.7, edgecolor='black')
    ax.set_title(f"VADER Compound Score Distribution -- {label}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Compound Score (-1 to +1)")
    ax.set_ylabel("Frequency")
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTrump - VADER Compound Score Stats:")
print(f"  Mean: {data_trump['vader_score'].mean():.3f}, Median: {data_trump['vader_score'].median():.3f}")
print(f"  Std Dev: {data_trump['vader_score'].std():.3f}")

print(f"\nBiden - VADER Compound Score Stats:")
print(f"  Mean: {data_biden['vader_score'].mean():.3f}, Median: {data_biden['vader_score'].median():.3f}")
print(f"  Std Dev: {data_biden['vader_score'].std():.3f}")

## Next Steps

This notebook completes the data preprocessing and exploratory analysis phase. The preprocessed datasets are now ready for advanced modeling:

- **04_roberta_finetuning.ipynb**: Fine-tunes `cardiffnlp/twitter-roberta-base` on 100K political tweets using VADER + NRCLex pseudo-labels for domain-specific emotion classification.

- **02_biden_sentiment_analysis.ipynb**: Applies the three-method pipeline (NRCLex + TextBlob + fine-tuned RoBERTa) on ~498K Biden tweets.

- **03_trump_sentiment_analysis.ipynb**: Applies the three-method pipeline on ~661K Trump tweets.

**Key artifacts generated in this notebook:**
- preprocessed_trump.csv
- preprocessed_biden.csv
- LDA models with interactive pyLDAvis topic cluster visualizations
- TextBlob and VADER sentiment scores

All visualizations and intermediate results are displayed above for reference.